# Trading Intelligent — CH01 — NB02
## Returns, Adjustments & Data Quality

**Auteur académique : Tawfik Masrour**  
**Édition : CH01 v2.1**

> **Question directrice :** Comment transformer une donnée brute en information statistiquement et économiquement légitime ?

### Ce que vous allez apprendre
- expliquer pourquoi une opération sur titres peut créer un faux rendement dans une série brute ;
- distinguer price return, total return, rendement simple et rendement logarithmique ;
- composer des rendements et construire une courbe de richesse ;
- interpréter une volatilité élémentaire sans l'assimiler à tout le risque ;
- diagnostiquer des défauts de données avant de construire features et target.

### Prérequis
NB01 est recommandé mais non obligatoire. Il suffit de connaître les opérations Python élémentaires et la notion de variation relative ; les rappels nécessaires sont fournis au moment utile.

### Comment utiliser ce notebook
Lisez la **question** avant d'exécuter la cellule Python. Formulez d'abord une réponse ou une prédiction, puis comparez-la à la sortie et à son interprétation. Les mini-défis servent à contrôler la compréhension, pas seulement la capacité à faire exécuter du code.

Ce notebook est un **laboratoire interactif de textbook** : question → intuition → expérience → observation → interprétation.

### 0. Charger la même source reproductible

Nous repartons de **TI_CH01_MARKET_V1** afin que les deux laboratoires racontent une histoire cohérente.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

DATASET_ID = "TI_CH01_MARKET_V1"
DATASET_FILENAME = "TI_CH01_Market_Data_v1.csv"
SEED = 20260929

# Palette commune au PDF et aux notebooks.
ACADEMIC_BLUE = "#173B57"
PETROL = "#2C7A7B"
WARM_GOLD = "#C79A3B"
MUTED_RED = "#A94B4B"
NEUTRAL_GRAY = "#66717D"
PALE_BLUE = "#EAF1F5"

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "semibold",
})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)


def generate_ti_ch01_market_v1(seed=SEED):
    """Reproduit exactement le jeu pédagogique synthétique TI_CH01_MARKET_V1."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=756)
    n = len(dates)
    vol = np.full(n, 0.010)
    vol[170:260] = 0.022
    vol[260:430] = 0.009
    vol[430:515] = 0.028
    vol[515:] = 0.012
    vol = pd.Series(vol).rolling(7, center=True, min_periods=1).mean().to_numpy()
    mu = np.full(n, 0.00020)
    mu[430:515] = -0.00035
    z = rng.standard_t(df=6, size=n) / np.sqrt(6/(6-2))
    log_ret = mu + vol * z
    close = 100 * np.exp(np.cumsum(log_ret))

    overnight = rng.normal(0, vol * 0.22)
    open_ = np.empty(n)
    open_[0] = 100 * np.exp(overnight[0])
    open_[1:] = close[:-1] * np.exp(overnight[1:])
    range_scale = np.maximum(vol, np.abs(log_ret)) * (0.8 + 0.6*rng.random(n))
    high = np.maximum(open_, close) * (1 + range_scale * (0.45 + 0.35*rng.random(n)))
    low = np.minimum(open_, close) * (1 - range_scale * (0.45 + 0.35*rng.random(n)))
    low = np.maximum(low, 0.01)
    volume = np.exp(16.3 + 11*np.abs(log_ret) + rng.normal(0, 0.28, n)).astype("int64")
    spread_bps = 3.0 + 85*vol + 35*np.abs(log_ret) + rng.gamma(1.5, 0.4, n)
    spread = close * spread_bps / 10000
    bid = close - spread/2
    ask = close + spread/2

    return pd.DataFrame({
        "date": dates, "symbol": "SYNTH-MKT",
        "open": open_, "high": high, "low": low, "close": close,
        "adjusted_close": close, "volume": volume, "bid": bid, "ask": ask,
    })


def locate_dataset(filename=DATASET_FILENAME):
    cwd = Path.cwd().resolve()
    roots = [cwd, *cwd.parents[:4]]
    subdirs = [Path("."), Path("datasets"), Path("data"), Path("resources")]
    seen = set()
    for root in roots:
        for sub in subdirs:
            candidate = (root / sub / filename).resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            if candidate.exists():
                return candidate
    return None

DATA_PATH = locate_dataset()
if DATA_PATH:
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    source_msg = f"{DATASET_ID} chargé depuis {DATA_PATH}"
else:
    df = generate_ti_ch01_market_v1()
    source_msg = f"{DATASET_ID} régénéré en mémoire (fallback déterministe)"

df = df.sort_values("date").reset_index(drop=True)
print(source_msg)
print("Dimensions :", df.shape)

### 1. Prix brut et split : une expérience volontairement incorrecte

**Question.** Que se passe-t-il si l'on calcule mécaniquement un rendement autour d'un split 2-pour-1 à partir du prix brut ?

In [ ]:
split=pd.DataFrame({
    'date':pd.bdate_range('2026-01-02',periods=8),
    'raw_close':[98,99,100,102,51,51.5,52,52.4]
})
split['raw_return']=split['raw_close'].pct_change()
display(split)
print('Rendement brut au split :',f"{100*split.loc[4,'raw_return']:.1f}%")

**Observation.** Le calcul donne environ `-50 %`. La formule est syntaxiquement correcte ; l'objet économique est faux pour mesurer la richesse, car le nombre d'actions a doublé.

### 2. Correction : construire une série comparable

Pour isoler le mécanisme, ajustons rétrospectivement les prix précédant le split par le facteur 2.

In [ ]:
split['adjusted_close']=split['raw_close'].astype(float)
split.loc[:3,'adjusted_close']=split.loc[:3,'raw_close']/2
split['adjusted_return']=split['adjusted_close'].pct_change()
display(split)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(split['date'],split['raw_close'],marker='o',color=MUTED_RED,label='prix brut')
ax.plot(split['date'],split['adjusted_close'],marker='o',color=PETROL,label='série ajustée')
ax.set_title("Un split peut créer un faux crash dans la série brute")
ax.legend(); ax.grid(alpha=.2); plt.show()

**À retenir.** « Prix ajusté » n'est pas synonyme de « meilleur prix ». Il est utile pour certaines comparaisons historiques ; une simulation d'exécution doit conserver les prix effectivement cotés et traiter explicitement les **opérations sur titres (OST)**.

### 3. Dividende : prix et richesse ne racontent pas exactement la même chose

**Question.** Si une action passe théoriquement de 100 à 95 lors d'un dividende de 5, l'investisseur a-t-il nécessairement perdu 5 % ?

In [ ]:
price_before=100
dividend=5
ex_div_theoretical=95
wealth_before=price_before
wealth_after=ex_div_theoretical+dividend
print('Richesse avant :',wealth_before)
print('Action après   :',ex_div_theoretical)
print('Cash dividende :',dividend)
print('Richesse simplifiée après :',wealth_after)

**Limite du raisonnement.** C'est un mécanisme simplifié, pas une loi exacte de marché : fiscalité, anticipation, microstructure et mouvements simultanés peuvent modifier le prix observé.

**Price return vs total return.** Le *price return* regarde la variation du prix seul ; le *total return* intègre aussi les distributions reçues, comme les dividendes. Un rendement basé uniquement sur le prix ne représente donc pas nécessairement toute la richesse de l'investisseur.

### 4. Rendement simple : calcul manuel avant `pct_change`

Pour `P0=100` et `P1=103`, le rendement simple vaut `(103/100)-1 = 3 %`. Vérifions la formule sur le dataset.

In [ ]:
p0=df.loc[0,'close']; p1=df.loc[1,'close']
r_manual=p1/p0-1
r_pandas=df['close'].pct_change().iloc[1]
print(f'P0={p0:.4f}, P1={p1:.4f}')
print('manuel :',r_manual)
print('pandas :',r_pandas)
print('écart  :',abs(r_manual-r_pandas))

### 5. +10 % puis -10 % : pourquoi on ne revient pas au point de départ

**Question.** Deux variations opposées de même taille s'annulent-elles ?

In [ ]:
capital=100
after_up=capital*1.10
after_down=after_up*0.90
print('Départ :',capital)
print('Après +10 % :',after_up)
print('Puis -10 %  :',after_down)
print('Rendement total :',f'{100*(after_down/capital-1):.2f}%')

**Interprétation.** Les rendements simples se composent multiplicativement. `+10 %` puis `-10 %` conduit à `-1 %`, pas à zéro.

### 6. Rendement logarithmique : l'additivité temporelle

Le rendement log est `log(Pt/Pt-1)` et suppose **`Pt > 0` et `Pt-1 > 0`**. Il est additif dans le temps, tandis que le rendement simple compose des facteurs `1+R`.

In [ ]:
assert (df['close']>0).all()
df['return_simple']=df['close'].pct_change()
df['return_log']=np.log(df['close']).diff()
comparison=pd.DataFrame({
    'simple':df['return_simple'],
    'log':df['return_log'],
    'abs_diff':(df['return_simple']-df['return_log']).abs()
})
display(comparison.head(8))
print('Écart médian |simple-log| :',comparison['abs_diff'].median())
print('Écart max |simple-log|    :',comparison['abs_diff'].max())

In [ ]:
grid=np.array([-0.20,-0.10,-0.02,0.02,0.10,0.20])
approx=pd.DataFrame({'R_simple':grid,'r_log':np.log1p(grid)})
approx['écart']=approx['R_simple']-approx['r_log']
display(approx.style.format('{:.4f}'))

**Observation.** L'approximation `log(1+R) ≈ R` est bonne pour de petites variations et se dégrade lorsque l'amplitude augmente.

### 7. De rendements à richesse et volatilité

**Question.** Comment passer d'une série de variations à une trajectoire de capital, puis à une mesure locale de dispersion ?

In [ ]:
df['wealth']=(1+df['return_simple'].fillna(0)).cumprod()
df['vol20_ann']=df['return_log'].rolling(20).std()*np.sqrt(252)
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(df['date'],df['wealth'],color=ACADEMIC_BLUE)
ax.set_title('Courbe de richesse'); ax.grid(alpha=.2); plt.show()
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(df['date'],100*df['vol20_ann'],color=WARM_GOLD)
ax.set_title('Volatilité glissante annualisée — 20 jours')
ax.set_ylabel('%'); ax.grid(alpha=.2); plt.show()

**Nuance.** La volatilité mesure ici la dispersion des rendements autour de leur niveau moyen. Elle est souvent utilisée comme **proxy de risque**, mais ne résume ni le risque de liquidité ni le risque de perte extrême. L'annualisation par `sqrt(252)` est une convention courante, pas une loi physique.

### 8. Data Quality Lab : injecter volontairement des défauts

**Question.** Un programme peut-il tourner sans erreur tout en manipulant une donnée économiquement incohérente ?

Oui. Créons une table volontairement corrompue : dates non triées, doublon, NaN, prix négatif, High incohérent et `Ask < Bid`.

In [ ]:
dirty=df.head(25).copy()
dirty=pd.concat([dirty,dirty.iloc[[5]]],ignore_index=True)
dirty.loc[2,'close']=np.nan
dirty.loc[7,'close']=-10
dirty.loc[9,'high']=min(dirty.loc[9,'open'],dirty.loc[9,'close'])-1
dirty.loc[11,'ask']=dirty.loc[11,'bid']-0.5
dirty=dirty.sample(frac=1,random_state=7).reset_index(drop=True)
display(dirty.head(10))

In [ ]:
def quality_report(x):
    numeric=['open','high','low','close','bid','ask']
    return pd.Series({
        'rows':len(x),
        'dates_sorted':bool(x['date'].is_monotonic_increasing),
        'duplicate_dates':int(x['date'].duplicated().sum()),
        'missing_values':int(x.isna().sum().sum()),
        'non_positive_prices':int((x[numeric]<=0).sum().sum()),
        'high_inconsistency':int((x['high']<x[['open','close']].max(axis=1)).fillna(False).sum()),
        'low_inconsistency':int((x['low']>x[['open','close']].min(axis=1)).fillna(False).sum()),
        'ask_below_bid':int((x['ask']<x['bid']).fillna(False).sum()),
    })

print('TABLE PROPRE')
display(quality_report(df).to_frame('value'))
print('TABLE VOLONTAIREMENT CORROMPUE')
display(quality_report(dirty).to_frame('value'))

**Interprétation.** Un rapport de qualité ne « nettoie » pas automatiquement : il localise les hypothèses violées. La correction dépend ensuite de la provenance et du sens économique du défaut.

### 9. Expérience volontairement incorrecte : transformer un trou en rendement nul

Remplacer aveuglément une valeur manquante par la dernière observation peut fabriquer artificiellement un rendement nul.

In [ ]:
toy=pd.DataFrame({'t':[0,1,2,3],'price':[100,101,np.nan,104]})
toy['return_raw']=toy['price'].pct_change(fill_method=None)
toy['price_ffill']=toy['price'].ffill()
toy['return_after_ffill']=toy['price_ffill'].pct_change(fill_method=None)
display(toy)

**Piège.** Le zéro créé à `t=2` n'est pas une observation de marché ; c'est une conséquence de la règle d'imputation. Elle peut être justifiée dans certains contextes, mais jamais sans convention explicite.

### 10. De la donnée aux features et à la target

**Question.** Comment transformer ce que l'on connaît au temps `t` en variables explicatives, tout en réservant le futur à la cible ?

In [ ]:
ml=df.copy()
ml['r_t']=ml['return_log']
ml['r_t_minus_1']=ml['return_log'].shift(1)
ml['vol20_t']=ml['return_log'].rolling(20).std()
ml['spread_bps_t']=10_000*(ml['ask']-ml['bid'])/((ml['ask']+ml['bid'])/2)
ml['target_up_next']=(ml['return_log'].shift(-1)>0).astype('Int64')
ml.loc[ml.index[-1],'target_up_next']=pd.NA
display(ml[['date','r_t','r_t_minus_1','vol20_t','spread_bps_t','target_up_next']].dropna().head())

**À retenir.** La target peut contenir le futur parce qu'elle représente ce que l'on veut apprendre à prédire. Les features au temps `t` doivent être disponibles au moment réel de la décision. Le chapitre 6 traitera systématiquement leakage et validation temporelle.

### 11. Micro-pont vers le Reinforcement Learning

Sans faire encore de RL, on peut déjà distinguer : **observation/state → action → conséquence**. Une action `+1` signifie ici acheter une unité, `0` rester neutre et `-1` vendre.

In [ ]:
i=300
state={
    'return_log_t':float(df.loc[i,'return_log']),
    'vol20_t':float(df['return_log'].rolling(20).std().iloc[i]),
    'position_before':0,
}
action=1
entry=df.loc[i,'ask']
exit_next=df.loc[i+1,'bid']
reward_one_step=exit_next-entry
print('State :',state)
print('Action:',action,'(achat)')
print(f'Exécution simplifiée : achat ask={entry:.4f}, sortie bid suivante={exit_next:.4f}')
print(f'Conséquence/PnL simplifié : {reward_one_step:.4f}')

**Limite.** Ce calcul n'est pas encore un environnement RL : il manque notamment MDP, politique, horizon, coûts complets et gestion de position. Il rend seulement tangible la chaîne décisionnelle.

### 12. Mini-défis

1. Reconstituez manuellement le rendement simple sur 5 jours et vérifiez avec `pct_change(5)`.
2. Ajoutez à `quality_report` un contrôle `high >= low` et un contrôle de volume négatif.
3. Identifiez les lignes exactes responsables de `Ask < Bid` et de l'incohérence OHLC.
4. Créez une feature `return_mean_5` qui n'utilise que le présent et le passé. Expliquez pourquoi une moyenne **centrée** serait dangereuse comme feature au temps `t`.
5. Pour un split 3-pour-1, proposez un exemple numérique équivalent au split 2-pour-1.

In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4

# Q5

### 13. Synthèse

- Un calcul peut être numériquement correct et économiquement faux.
- Les opérations sur titres obligent à distinguer prix coté, ajustement et richesse.
- Price return et total return ne décrivent pas exactement le même objet.
- Rendements simple et logarithmique ont des propriétés de composition différentes.
- La volatilité est une mesure de dispersion souvent utilisée comme proxy de risque, pas une mesure exhaustive du risque.
- La qualité des données se vérifie par des contrats explicites, pas par intuition.
- Une feature appartient à l'information disponible ; une target peut représenter le futur à prédire.
- Le passage vers le RL commence par observation, action et conséquence — sans anticiper les chapitres dédiés.

### Checklist de maîtrise

Avant de considérer NB02 acquis, vérifiez que vous pouvez répondre **oui** sans relire le notebook :

- [ ] Je peux expliquer pourquoi un split peut créer un faux rendement dans une série brute.
- [ ] Je peux distinguer price return et total return.
- [ ] Je peux calculer et composer un rendement simple, puis expliquer l'additivité du rendement logarithmique.
- [ ] Je sais que le rendement logarithmique suppose des prix strictement positifs.
- [ ] Je peux expliquer ce que mesure la volatilité et pourquoi elle ne résume pas tout le risque.
- [ ] Je peux reconnaître dates non triées, doublons, NaN, prix impossibles, incohérences OHLC et `Ask < Bid`.
- [ ] Je peux expliquer pourquoi une règle d'imputation peut fabriquer une observation qui n'a jamais existé.
- [ ] Je peux distinguer feature disponible au temps `t` et target future à prédire.